# Step 5: Delta Lake and ACID Transactions

Plain Parquet files present a limitation: if a write fails partway through, or two processes write concurrently, a reader may encounter a partially written, inconsistent state, since no transactional guarantee exists.

**Delta Lake** addresses this by adding a transaction log (`_delta_log/`) on top of ordinary Parquet files. Every write — insert, update, or delete — becomes a single atomic JSON log entry. Readers always see a consistent snapshot, and because prior data files are never overwritten, it is possible to time-travel to any previous version. This same idea — plain data files plus a transaction log — is not specific to Parquet; other table formats (such as Apache Iceberg and Apache Hudi) apply it on top of other file formats too, and Delta Lake itself simply standardizes on Parquet as its data file format.

This notebook uses the `deltalake` Python package (built on [delta-rs](https://github.com/delta-io/delta-rs)), which requires neither Spark nor a JVM.

In [ ]:
import os
from pathlib import Path

while not (Path.cwd() / "requirements.txt").exists():
    os.chdir("..")
print("Working directory:", Path.cwd())

## Version 0: Initial Write

January 2026 is aggregated into a small daily-revenue-per-region table and written as a new Delta table.

In [ ]:
import shutil

import duckdb
from deltalake import DeltaTable, write_deltalake

DELTA_PATH = "lake/delta/sales_delta"

# Clean slate so this notebook is re-runnable
shutil.rmtree(DELTA_PATH, ignore_errors=True)

con = duckdb.connect()

jan = con.sql("""
    SELECT date, region, SUM(revenue) AS revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026 AND monat = 1
    GROUP BY date, region
""").to_arrow_table()

write_deltalake(DELTA_PATH, jan, mode="overwrite")
print(f"Version 0 written: {len(jan)} rows")

## Version 1: Append

February data is now added. This constitutes a new transaction; the January data files remain untouched.

In [ ]:
feb = con.sql("""
    SELECT date, region, SUM(revenue) AS revenue
    FROM 'lake/verkauf/**/*.parquet'
    WHERE jahr = 2026 AND monat = 2
    GROUP BY date, region
""").to_arrow_table()

write_deltalake(DELTA_PATH, feb, mode="append")
print(f"Version 1 written (append): {len(feb)} rows")

## Version 2: Update

A correction is simulated, such as a currency adjustment for Zurich. Plain Parquet has no `UPDATE` operation; Delta Lake provides one, and it is transactional — either the entire update is applied, or none of it is.

In [ ]:
dt = DeltaTable(DELTA_PATH)
dt.update(predicate="region = 'Zurich'", updates={"revenue": "revenue * 1.1"})
print("Version 2 written (update)")

## Inspecting `_delta_log/`

Each write produces exactly one JSON log file, viewable in the file explorer or listed directly below.

In [ ]:
log_dir = Path(DELTA_PATH) / "_delta_log"
for f in sorted(log_dir.glob("*.json")):
    print(f.name)

In [ ]:
import json

first_log = sorted(log_dir.glob("*.json"))[0]
for line in first_log.read_text().splitlines():
    entry = json.loads(line)
    print(list(entry.keys()))

Each line represents one *action*: `add` registers a new Parquet data file as part of the table, `commitInfo` records the operation performed and its timestamp, and `metaData`/`protocol` describe the schema and format version. A snapshot of the table at any given version is obtained by replaying every log entry up to that version and determining which files are currently `add`ed.

## Transaction History

`DeltaTable.history()` retrieves `commitInfo` entries as a structured list, constituting the table's audit trail.

In [ ]:
dt = DeltaTable(DELTA_PATH)
for entry in dt.history():
    print(entry.get("version"), "-", entry.get("operation"))

## Time Travel

Loading an earlier version reads the log only up to that point; the update applied afterward is not visible from version 0's perspective, even though no data was deleted from disk.

In [ ]:
dt_v0 = DeltaTable(DELTA_PATH, version=0)
v0_total = dt_v0.to_pandas()["revenue"].sum()

dt_latest = DeltaTable(DELTA_PATH)
latest_total = dt_latest.to_pandas()["revenue"].sum()

print(
    "Version 0 (January only, pre-correction) total revenue: "
    f"{v0_total:,.2f}"
)
print(
    "Latest version (Jan+Feb, post-correction) total revenue: "
    f"{latest_total:,.2f}"
)

## Summary

This progression mirrors the structure of the course as a whole, compressed into a single table:

1. **File format** (Steps 1–2, `01`/`02`) — CSV was converted to compressed, columnar Parquet.
2. **Storage organization** (Steps 2–4, `02`/`04`) — files were organized into partitions, then migrated to object storage.
3. **The query engine** (Steps 3–4, `03`/`04`) — DuckDB queried the same files using schema-on-read, predicate pushdown, and column pruning.
4. **Transactions** (Step 5, `05`, this notebook) — Delta Lake introduced the capability plain files lacked.

Each stage addressed a limitation exposed by the previous one. This progression, rather than any single tool, is the central lesson of the course.